In [2]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import tensorflow as tf
from collections import Counter
import pickle
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.applications.xception import Xception, preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, Add
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import GaussianNoise

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ZIP_PATH = '/content/drive/MyDrive/caption_data.zip' # main data
WORKING_DIR = '/content/caption_data'

if not os.path.exists(WORKING_DIR):
    !unzip -q {ZIP_PATH} -d {WORKING_DIR}
    print("Unzipped successfully!")

IMAGE_DIR = os.path.join(WORKING_DIR, 'Images')
CAPTIONS_FILE = os.path.join(WORKING_DIR, 'captions.txt')

Unzipped successfully!


In [5]:
def load_and_clean_captions(mapping_file):
    df = pd.read_csv(mapping_file)

    mapping = {}

    for i, row in df.iterrows():
        img_name = row['image']
        caption = str(row['caption'])

        # lowercase and remove special characters
        caption = caption.lower()
        caption = "".join([char for char in caption if char.isalpha() or char.isspace() or char.isnumeric()]) #to be able to distinguish letters, spaces and nnumbers
        # Adding the required start, end tokens for the sequence model, for the model to undestand when does the caption start and when does it end
        caption = f"startseq {caption} endseq"

        if img_name not in mapping:
            mapping[img_name] = []

        mapping[img_name].append(caption)

    return mapping

#dictionary - image: caption
captions_dict = load_and_clean_captions(CAPTIONS_FILE)
print(f"Loaded {len(captions_dict)} unique images.")

Loaded 8091 unique images.


In [9]:
all_captions = []

for caps in captions_dict.values():
    for c in caps:
        all_captions.extend(c.split())

word_freq = Counter(all_captions)

EXCLUDE_WORDS = {
    "a", "an", "the",
    "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "at", "for", "with", "from",
    "and", "or", "but",
    "he", "she", "it", "they", "his", "her", "their",
    "this", "that", "these", "those",
    "startseq", "endseq"
}

filtered_words = [
    (word, count)
    for word, count in word_freq.items()
    if word not in EXCLUDE_WORDS and len(word) > 2
]

top_30_content = sorted(
    filtered_words,
    key=lambda x: x[1],
    reverse=True
)[:30]

top_30_content



[('dog', 8136),
 ('man', 7265),
 ('two', 5638),
 ('white', 3940),
 ('black', 3832),
 ('boy', 3581),
 ('woman', 3402),
 ('girl', 3328),
 ('wearing', 3062),
 ('people', 2883),
 ('water', 2783),
 ('red', 2672),
 ('young', 2630),
 ('brown', 2563),
 ('blue', 2268),
 ('dogs', 2125),
 ('running', 2073),
 ('through', 2032),
 ('playing', 2008),
 ('while', 1968),
 ('down', 1825),
 ('shirt', 1806),
 ('standing', 1787),
 ('ball', 1779),
 ('little', 1768),
 ('grass', 1622),
 ('child', 1545),
 ('person', 1542),
 ('snow', 1492),
 ('jumping', 1472)]

In [12]:
total_captions = sum(len(caps) for caps in captions_dict.values())
total_words = sum(word_freq.values())

print(f"Total images: {len(captions_dict)}")
print(f"Total captions: {total_captions}")
print(f"Total words: {total_words}")
print(f"Vocabulary size: {len(word_freq)}")

categories = {
    'outdoor_scene': ['field', 'grass', 'park', 'beach', 'street', 'road', 'outside', 'outdoor'],
    'indoor_scene': ['room', 'kitchen', 'bedroom', 'living', 'indoor', 'inside', 'ceiling', 'floor', 'wall'],
    'people': ['man', 'woman', 'boy', 'girl', 'person', 'people', 'child', 'children'],
    'animals': ['dog', 'dogs', 'cat', 'cats', 'horse', 'bird'],
    'actions': ['running', 'playing', 'standing', 'jumping', 'sitting', 'walking', 'riding'],
    'food': ['food', 'plate', 'eating', 'meal', 'dish', 'restaurant', 'dining', 'pizza', 'pasta'],
    'furniture': ['couch', 'sofa', 'chair', 'table', 'bed', 'desk'],
    'colors': ['white', 'black', 'red', 'brown', 'blue', 'green', 'yellow'],
    'water_related': ['water', 'ocean', 'sea', 'beach', 'swimming', 'surfing'],
    'winter': ['snow', 'snowy', 'winter', 'ice']
}

category_counts = {}
for category, keywords in categories.items():
    count = sum(word_freq.get(word, 0) for word in keywords)
    category_counts[category] = count

print("\nCategory Word Frequencies:")
for category, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_words) * 100
    print(f"{category:20s}: {count:6,} words ({percentage:.2f}%)")

print(" vocabulary check: ")

missing_indoor = ['room', 'kitchen', 'bedroom', 'living', 'ceiling', 'interior', 'furniture', 'couch', 'sofa']
missing_food = ['food', 'plate', 'dish', 'meal', 'eating', 'restaurant', 'dining', 'pasta', 'pizza', 'bread']
missing_perspective = ['aerial', 'overhead', 'above', 'crowd', 'crowded']

print("\nindoor vocabulary:")
for word in missing_indoor:
    count = word_freq.get(word, 0)
    if count == 0:
        print(f"'{word}' - MISSING (0 occurrences)")
    else:
        print(f"'{word}' - {count} occurrences")

print("\nfood vocabulary:")
for word in missing_food:
    count = word_freq.get(word, 0)
    if count == 0:
        print(f" '{word}' - MISSING (0 occurrences)")
    else:
        print(f"'{word}' - {count} occurrences")

print("\nPerspective/Viewpoint Vocabulary:")
for word in missing_perspective:
    count = word_freq.get(word, 0)
    if count == 0:
        print(f" '{word}' - MISSING (0 occurrences)")
    else:
        print(f" '{word}' - {count} occurrences")

# categories %
print(" summary:")

outdoor_total = category_counts['outdoor_scene'] + category_counts['water_related'] + category_counts['winter']
people_animals = category_counts['people'] + category_counts['animals']
indoor_food = category_counts['indoor_scene'] + category_counts['food'] + category_counts['furniture']

total_analyzed = outdoor_total + people_animals + indoor_food

print(f"\nOutdoor scenes: {outdoor_total:,} words ({outdoor_total/total_analyzed*100:.1f}%)")
print(f"People & Animals: {people_animals:,} words ({people_animals/total_analyzed*100:.1f}%)")
print(f"Indoor & Food: {indoor_food:,} words ({indoor_food/total_analyzed*100:.1f}%)")

common_actions = ['running', 'playing', 'standing', 'jumping', 'sitting', 'walking',
                  'riding', 'wearing', 'holding', 'looking', 'eating', 'drinking']

print("\nAction word frequencies:")
for action in common_actions:
    count = word_freq.get(action, 0)
    print(f"  {action:15s}: {count:5,} occurrences")

Total images: 8091
Total captions: 40455
Total words: 517415
Vocabulary size: 8830

Category Word Frequencies:
people              : 24,702 words (4.77%)
colors              : 17,713 words (3.42%)
actions             : 10,780 words (2.08%)
animals             : 10,762 words (2.08%)
outdoor_scene       :  6,711 words (1.30%)
water_related       :  4,864 words (0.94%)
winter              :  2,250 words (0.43%)
indoor_scene        :  1,136 words (0.22%)
furniture           :    686 words (0.13%)
food                :    402 words (0.08%)
 vocabulary check: 

indoor vocabulary:
'room' - 132 occurrences
'kitchen' - 45 occurrences
'bedroom' - 5 occurrences
'living' - 20 occurrences
'ceiling' - 8 occurrences
'interior' - 2 occurrences
'furniture' - 10 occurrences
'couch' - 102 occurrences
'sofa' - 20 occurrences

food vocabulary:
'food' - 92 occurrences
'plate' - 36 occurrences
'dish' - 11 occurrences
'meal' - 14 occurrences
'eating' - 144 occurrences
'restaurant' - 85 occurrences
'dining' - 

In [5]:
# Loading Xception without the top classification layer
# It was pre-trained on ImageNet (millions of images)
model = Xception(include_top=False, pooling='avg')

"""
feature extraction - Convert the image into numbers that describe its visual content

include_top=False -
The top layer is a classifier like - cat, dog. which we do not want.
We want visual features, not labels

pooling='avg'- (On avg which part of the picture matched best to our kernel)
Converts feature maps into one fixed-length vector
This makes it easy to feed into a text model later

This function:
Goes through all images
Converts each image into a feature vector
Stores the result in a dictionary
"""
def extract_features(directory, mapping, augment=True):

    features = {}
    for img_name in tqdm(mapping.keys()):
        # Reads the image from disk, resize requires by the model
        img_path = os.path.join(directory, img_name)
        image = load_img(img_path, target_size=(299, 299))

        # Convert to array and preprocess
        image = img_to_array(image) # Image → NumPy array, (height, width, RGB channels)
        image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2])) # neural networks expect input as batches => shape now: (1, 299, 299, 3)
        image = preprocess_input(image) # Scales pixel values


        # Extract features - compressed numeric description of the image
        feature = model.predict(image, verbose=0)
        features[img_name] = feature # Dictiorary structure for each picture

    return features

features = extract_features(IMAGE_DIR, captions_dict, augment=False)

with open('image_features.pkl', 'wb') as f:
    pickle.dump(features, f)

print("Features extracted and saved to image_features.pkl")

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


100%|██████████| 8091/8091 [11:51<00:00, 11.37it/s]


Features extracted and saved to image_features.pkl


In [6]:
all_captions = []
for key, caps in captions_dict.items():
    for cap in caps:
        all_captions.append(cap)

# vocab limit to top 10,000 words
VOCAB_LIMIT = 10000
tokenizer = Tokenizer(num_words=VOCAB_LIMIT, filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~', oov_token="<unk>")
tokenizer.fit_on_texts(all_captions)

#either the limit we set, or the actual number of words found
vocab_size = min(VOCAB_LIMIT, len(tokenizer.word_index) + 1)
max_length = max(len(cap.split()) for cap in all_captions)

print(f"Vocabulary Size: {vocab_size}")
print(f"Max Caption Length: {max_length}")

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

<>:8: SyntaxWarning: invalid escape sequence '\]'
<>:8: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipython-input-2824397696.py:8: SyntaxWarning: invalid escape sequence '\]'
  tokenizer = Tokenizer(num_words=VOCAB_LIMIT, filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~', oov_token="<unk>")


Vocabulary Size: 8832
Max Caption Length: 38


In [7]:
"""The data generator transforms each image caption into many
  (image + partial caption → next word) training samples,
  pads text sequences, one-hot encodes targets,
  and streams batches efficiently to the model."""

def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0
    while 1:
        for key in data_keys:
            n += 1
            captions = mapping[key]
            feature = features[key][0]
            # captions: list of captions for this image
            # feature: 2048-dimensional vector from Xception
            # [0] removes batch dimension (1, 2048 → 2048)

            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                # for example
                # startseq a dog runs endseq
                # ↓
                # [1, 5, 23, 89, 2]

                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i] # given the image + words so far, predict the next word

                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    # nns require fixed-length inputs
                    # Shorter sequences are padded with zeros at the end

                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0] # One-hot encode output word,
                                                                  # example, out_seq = 23 -> [0, 0, 0, ..., 1, ..., 0]  # length = vocab_size

                    X1.append(feature)
                    X2.append(in_seq)
                    y.append(out_seq)

            if n == batch_size:
                yield (np.array(X1), np.array(X2)), np.array(y)
                X1, X2, y = list(), list(), list()
                n = 0
                # X1 → image features
                # X2 → partial text sequences
                # y → next word (one-hot encoded)

                """
                Once batch_size images are processed:
                Convert lists → NumPy arrays
                Yield batch to the model
                Reset batch containers
                """



In [8]:
""" model has two inputs:
Image features (from Xception),
Partial caption text (words so far)
output - Next word prediction

encoder–decoder model with late fusion."""

# Feature Extractor (Encoder)
inputs1 = Input(shape=(2048,))
fe1 = GaussianNoise(0.1)(inputs1) # GaussianNoise helps prevent overfitting to specific image features
fe2 = Dropout(0.3)(fe1)
fe3 = Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(fe2) # L2 Regularization helps keep weights small


# Sequence Processor (Decoder)
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 128, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(128, dropout=0.3, recurrent_dropout=0.3)(se2)

# The Merge
decoder1 = Add()([fe3, se3])
decoder2 = Dense(128, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    optimizer='adam'
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise      │ (None, 2048)      │          0 │ input_layer_1[0]… │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 38, 128)   │  1,130,496 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ gaussian_noise[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 38, 128)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 38)        │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    262,272 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │    131,584 │ dropout_1[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 128)       │          0 │ dense[0][0],      │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ add_12[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8832)      │  1,139,328 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,680,192 (10.22 MB)

 Trainable params: 2,680,192 (10.22 MB)

 Non-trainable params: 0 (0.00 B)